**1. Design and implement a neural based network for generating word embedding for words in a document corpus.**

Concept Overview
Word embeddings are vector representations of words that capture semantic relationships. A common approach to generate them is using neural architectures like:
*   Word2Vec (CBOW or Skip-gram)
*   GloVe
*   FastText
*   Transformer-based models (e.g., BERT — contextual embeddings)

For implementation from scratch, Word2Vec's Skip-gram is a great starting point due to its simplicity and effectiveness.

Design Outline: Skip-Gram Neural Network

Objective

Train a model to predict context words given a target word.

Input
*   A large corpus of text
*   Vocabulary derived from the corpus

Steps

1.	Data Preprocessing
  *   Tokenize text
  *   Build vocabulary and assign word IDs
  *   Generate training pairs (target_word, context_word)

2.	Network Architecture
  *   Input layer: One-hot vector of target word
  *   Hidden layer: Projection to n-dimensional embedding space
  *   Output layer: Softmax over vocabulary to predict context word
3.	Loss Function
  *   Negative sampling (efficient approximation to full softmax)



### Understanding the SkipGram Model (`SkipGram` class)

This cell defines our neural network, `SkipGram`. At its core, it's designed to learn word embeddings. Think of a word embedding as a numerical fingerprint for each word, where words with similar meanings have similar fingerprints.

#### How it works:

1.  **`nn.Embedding(vocab_size, embed_dim)`**: This is the heart of our word embedding. Imagine a giant lookup table where each row corresponds to a word in our vocabulary, and each column is a dimension of its embedding. When you input a word's ID, it 'looks up' and returns its unique embedding vector. Initially, these vectors are random, but during training, they adjust to capture meaning.

    *   `vocab_size`: The total number of unique words in our dictionary.
    *   `embed_dim`: The size of the numerical fingerprint (vector) for each word. A larger dimension can capture more nuanced relationships, but also requires more data to train effectively.

2.  **`nn.Linear(embed_dim, vocab_size)`**: This is a standard neural network layer. After getting a word's embedding, this layer tries to predict *which other words* are likely to appear in its context. It takes the `embed_dim` vector and transforms it into a `vocab_size` vector, where each element represents the probability of a word from the vocabulary being a context word.

#### Visualizing the Architecture:

Imagine the network taking a target word (e.g., "love"), looking up its embedding, and then using that embedding to guess other words that might appear around it (e.g., "movies", "reading").

```mermaid
graph TD
    A[Target Word ID] --> B(nn.Embedding Layer)
    B --> C{Word Embedding Vector}
    C --> D(nn.Linear Layer)
    D --> E[Output Probabilities for Context Words]
```

During training, if the model correctly predicts a context word, the embedding for the target word (and related context words) gets adjusted slightly to make similar predictions easier in the future.

In [ ]:
#!pip install torch
import torch
import torch.nn as nn
import torch.optim as optim
from collections import defaultdict
import random
import numpy as np

# Define SkipGram model
class SkipGram(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super(SkipGram, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embed_dim)
        self.output = nn.Linear(embed_dim, vocab_size)

    def forward(self, input):
        embed = self.embeddings(input)
        out = self.output(embed)
        return out

# Sample training loop (mocked data)
def train(model, data, epochs=100, lr=0.01):
    optimizer = optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        total_loss = 0
        for target, context in data:
            model.zero_grad()
            output = model(torch.tensor([target]))
            loss = loss_fn(output, torch.tensor([context]))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f'Epoch {epoch}, Loss: {total_loss:.4f}')


### Data Preprocessing

### Data Preprocessing: Turning Text into Numbers

Computers don't understand words directly; they understand numbers. This section converts our raw text corpus into a numerical format that the neural network can process.

#### 1. Tokenization

We break down sentences into individual words, called 'tokens'. For example, "I love movies" becomes `['i', 'love', 'movies']`.

#### 2. Building Vocabulary

We collect all unique words from our entire corpus to create a vocabulary. Each unique word is then assigned a unique numerical ID. This `word_to_idx` (word to index) mapping allows us to represent any word as a number.

#### Example of Tokenization and Vocabulary:

If we have the sentence: "I love to watch movies in the evening"

It would be tokenized into: `['i', 'love', 'to', 'watch', 'movies', 'in', 'the', 'evening']`

Then, each unique word gets an ID. For example, from the output above:

*   `'books'`: `3`
*   `'love'`: `11`
*   `'morning'`: `12`
*   `'movies'`: `13`

And `vocab_size` tells us how many unique words we have in total.

In [ ]:
corpus = [
    "I love to watch movies in the evening",
    "I love to read books in the morning",
    "movies and books are great for relaxation",
    "reading is a great way to learn new things"
]

# 1. Tokenize text and build vocabulary
word_to_idx = {}
idx_to_word = {}
vocab = set()

for sentence in corpus:
    for word in sentence.lower().split():
        vocab.add(word)

for i, word in enumerate(sorted(list(vocab))):
    word_to_idx[word] = i
    idx_to_word[i] = word

vocab_size = len(vocab)
print(f"Vocabulary size: {vocab_size}")
print(f"Word to index mapping: {word_to_idx}")


Vocabulary size: 23
Word to index mapping: {'a': 0, 'and': 1, 'are': 2, 'books': 3, 'evening': 4, 'for': 5, 'great': 6, 'i': 7, 'in': 8, 'is': 9, 'learn': 10, 'love': 11, 'morning': 12, 'movies': 13, 'new': 14, 'read': 15, 'reading': 16, 'relaxation': 17, 'the': 18, 'things': 19, 'to': 20, 'watch': 21, 'way': 22}


### Generate Training Pairs (Target, Context)

### Generating Training Pairs: Learning from Context

For the Skip-Gram model to learn word relationships, we need to show it examples of words that appear together. This cell creates these "training pairs" consisting of a `(target_word, context_word)`.

#### How it works:

1.  **Sliding Window**: We slide a 'window' of a specified `window_size` across each sentence. For every word in the sentence (the `target_word`), we consider other words within this window as `context_words`.

2.  **Excluding Self**: A word is never considered its own context, so the `target_word` itself is excluded from its context pairs.

#### Visualizing Training Pair Generation:

Let's take the sentence "I love to watch movies in the evening" and a `window_size = 2`.
*   If "watch" is the target word (ID 21), with a `window_size` of 2, its context words are "love", "to", "movies", "in".
*   So, we generate pairs like `(21, 11)`, `(21, 20)`, `(21, 13)`, `(21, 8)`.

The output confirms we generated `104` such pairs from our small corpus, ready to teach the model about word relationships.

In [ ]:
def generate_pairs(corpus, word_to_idx, window_size=2):
    data = []
    for sentence in corpus:
        tokens = sentence.lower().split()
        indices = [word_to_idx[word] for word in tokens]
        for i, target_idx in enumerate(indices):
            for j in range(max(0, i - window_size), min(len(indices), i + window_size + 1)):
                if i != j:
                    context_idx = indices[j]
                    data.append((target_idx, context_idx))
    return data


training_data = generate_pairs(corpus, word_to_idx, window_size=2)
print(f"Number of training pairs: {len(training_data)}")
print(f"Sample training pairs: {training_data[:5]}")


Number of training pairs: 104
Sample training pairs: [(7, 11), (7, 20), (11, 7), (11, 20), (11, 21)]


### Instantiate and Train the SkipGram Model

### Training the SkipGram Model

This is where the magic happens! The neural network learns the word embeddings by iteratively adjusting them based on the training pairs we generated.

#### The Training Loop:

1.  **`epochs`**: This is the number of times the model will go through the entire `training_data`.
2.  **`optimizer = optim.SGD(...)`**: This is the algorithm that adjusts the model's parameters (our word embeddings) to reduce the error. SGD (Stochastic Gradient Descent) is a common choice.
3.  **`loss_fn = nn.CrossEntropyLoss()`**: This function measures how 'wrong' the model's predictions are. Our goal during training is to minimize this loss.

    *   For each `(target, context)` pair, the model predicts a probability distribution over all vocabulary words. The loss function compares this prediction to the actual `context_word` and gives a high score if the prediction was far off, and a low score if it was accurate.

#### What the Loss Means:

In the output, you see `Loss: 349.8397` for `Epoch 0`, gradually decreasing to much smaller values (truncated in the example). This decreasing loss indicates that our model is learning! It's getting better at predicting context words given a target word, and in doing so, its internal `word_embeddings` are becoming more meaningful.

#### What are `word_embeddings`?

After training, `model.embeddings.weight.detach().numpy()` gives us the final word embeddings. Each word now has a unique vector of 10 numbers (since `embed_dim = 10`) that represents its meaning based on the contexts it appeared in. Words that appear in similar contexts will have similar embedding vectors.

In [ ]:
embed_dim = 10
model = SkipGram(vocab_size, embed_dim)

print("Starting training...")
train(model, training_data, epochs=500, lr=0.01)
print("Training complete!")

# Get word embeddings
word_embeddings = model.embeddings.weight.detach().numpy()
print("\nWord Embeddings:")
for word, idx in word_to_idx.items():
    print(f"{word}: {word_embeddings[idx]}")


Starting training...
Epoch 0, Loss: 344.5491
Epoch 1, Loss: 329.7878
Epoch 2, Loss: 317.5556
Epoch 3, Loss: 307.4899
Epoch 4, Loss: 299.2011
Epoch 5, Loss: 292.2942
Epoch 6, Loss: 286.4257
Epoch 7, Loss: 281.3335
Epoch 8, Loss: 276.8331
Epoch 9, Loss: 272.7974
Epoch 10, Loss: 269.1370
Epoch 11, Loss: 265.7872
Epoch 12, Loss: 262.6987
Epoch 13, Loss: 259.8334
Epoch 14, Loss: 257.1604
Epoch 15, Loss: 254.6547
Epoch 16, Loss: 252.2957
Epoch 17, Loss: 250.0660
Epoch 18, Loss: 247.9510
Epoch 19, Loss: 245.9382
Epoch 20, Loss: 244.0171
Epoch 21, Loss: 242.1784
Epoch 22, Loss: 240.4142
Epoch 23, Loss: 238.7178
Epoch 24, Loss: 237.0833
Epoch 25, Loss: 235.5054
Epoch 26, Loss: 233.9796
Epoch 27, Loss: 232.5022
Epoch 28, Loss: 231.0697
Epoch 29, Loss: 229.6790
Epoch 30, Loss: 228.3277
Epoch 31, Loss: 227.0134
Epoch 32, Loss: 225.7341
Epoch 33, Loss: 224.4880
Epoch 34, Loss: 223.2736
Epoch 35, Loss: 222.0895
Epoch 36, Loss: 220.9343
Epoch 37, Loss: 219.8071
Epoch 38, Loss: 218.7067
Epoch 39, Loss

### Inference Example: Finding Similar Words

### Inference: Finding Similar Words

Now that our model has learned numerical representations (embeddings) for each word, we can use these embeddings to find out which words are 'similar' to each other. This is a common way to demonstrate the quality of the learned embeddings.

#### How it works:

1.  **Cosine Similarity**: We use a metric called **cosine similarity**. Imagine each word embedding as a point in a multi-dimensional space. Cosine similarity measures the cosine of the angle between two embedding vectors. If two vectors point in roughly the same direction (small angle, cosine close to 1), the words are considered similar. If they point in very different directions (large angle, cosine close to -1 or 0), they are dissimilar.
2.  **`find_similar_words` function**: This function takes a `word`, gets its embedding, and then calculates the cosine similarity between that word's embedding and *every other word's embedding* in the vocabulary. It then sorts these similarities and returns the `top_n` most similar words (excluding the word itself).

#### Interpreting the Results:

*   **`books`**: `for`, `morning`, `evening`, `watch`, `in`. While not all are direct synonyms, these words appeared in similar contexts to 'books' in our small corpus. The highest similarity is to 'for' (0.3938), which makes sense if sentences like "books **for** relaxation" were present.
*   **`love`**: `new`, `watch`, `in`, `great`, `learn`. This suggests 'love' was associated with activities and general positive descriptions.
*   **`morning`**: `movies`, `the`, `evening`, `watch`, `in`. This is interesting because 'morning' and 'evening' are antonyms, but they both appear in the context of time, and likely alongside words like 'watch' and 'movies' in our corpus. The high similarity between 'morning' and 'movies' (0.6779) is directly influenced by the sentence "I love to watch movies in the evening" and "I love to read books in the morning", where `movies` and `morning` are both contexts for `i love to watch`. This shows how context-specific the embeddings are.

This inference step shows that our small Skip-Gram model, even with limited data, has started to capture some meaningful relationships between words!

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def find_similar_words(word, word_to_idx, idx_to_word, word_embeddings, top_n=5):
    if word not in word_to_idx:
        print(f"'{word}' not in vocabulary.")
        return

    word_idx = word_to_idx[word]
    word_vec = word_embeddings[word_idx].reshape(1, -1)

    similarities = cosine_similarity(word_vec, word_embeddings)[0]

    # Sort by similarity and get top_n words (excluding the word itself)
    sorted_indices = np.argsort(similarities)[::-1]

    print(f"\nWords most similar to '{word}':")
    count = 0
    for i in sorted_indices:
        if idx_to_word[i] != word:
            print(f"  {idx_to_word[i]}: {similarities[i]:.4f}")
            count += 1
        if count >= top_n:
            break

# Example usage:
find_similar_words('books', word_to_idx, idx_to_word, word_embeddings)
find_similar_words('love', word_to_idx, idx_to_word, word_embeddings)
find_similar_words('morning', word_to_idx, idx_to_word, word_embeddings)



Words most similar to 'books':
  for: 0.4109
  watch: 0.3539
  evening: 0.2879
  and: 0.2280
  morning: 0.2249

Words most similar to 'love':
  learn: 0.5231
  i: 0.3710
  new: 0.2877
  watch: 0.2350
  read: 0.1972

Words most similar to 'morning':
  evening: 0.6744
  read: 0.4219
  are: 0.3601
  new: 0.3138
  i: 0.2562
